In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Lowercase


In [5]:
df['review'] = df['review'].str.lower()

In [6]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


Remove HTML Tags

In [7]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'',text)

In [8]:
df['review'].apply(remove_html_tags)

0        one of the other reviewers has mentioned that ...
1        a wonderful little production. the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically there's a family where a little boy ...
4        petter mattei's "love in the time of money" is...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: object

remove url

In [12]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'',text)

In [13]:
text = 'Check out my notebook https://www.kaggle.com/code/tajshuvo/notebook0ccaca528c/edit'

In [14]:
remove_url(text)

'Check out my notebook '

Removing Punctuation

In [15]:
import string,time

string.punctuation


'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [16]:
exclude = string.punctuation

In [17]:
def remove_punc(text):#slowwwwww
    for char in exclude:
        text = text.replace(char,'')
    return text

In [18]:
text = 'String, With. Punctuation? '

In [19]:
remove_punc(text)

'String With Punctuation '

In [20]:
def remove_punc(text):
    return text.translate(str.maketrans('','',exclude))

In [21]:
df['review'] = df['review'].apply(remove_punc)

In [22]:
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production br br the filmin...
2        i thought this was a wonderful way to spend ti...
3        basically theres a family where a little boy j...
4        petter matteis love in the time of money is a ...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    i am a catholic taught in parochial elementary...
49998    im going to have to disagree with the previous...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: object

Chat word Treatment

In [24]:
chat_word = {
    "A3": "Anytime, Anywhere, Anyplace",
    "ADIH": "Another Day In Hell",
    "AFK": "Away From Keyboard",
    "AFAIK": "As Far As I Know",
    "ASAP": "As Soon As Possible",
    "ASL": "Age, Sex, Location",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "BAE": "Before Anyone Else",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRUH": "Bro",
    "BRT": "Be Right There",
    "BSAAW": "Big Smile And A Wink",
    "BTW": "By The Way",
    "BWL": "Bursting With Laughter",
    "CSL": "Can’t Stop Laughing",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "DM": "Direct Message",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FIMH": "Forever In My Heart",
    "FOMO": "Fear Of Missing Out",
    "FR": "For Real",
    "FWIW": "For What It's Worth",
    "FYP": "For You Page",
    "FYI": "For Your Information",
    "G9": "Genius",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GMTA": "Great Minds Think Alike",
    "GN": "Good Night",
    "GOAT": "Greatest Of All Time",
    "GR8": "Great!",
    "HBD": "Happy Birthday",
    "IC": "I See",
    "ICQ": "I Seek You",
    "IDC": "I Don’t Care",
    "IDK": "I Don't Know",
    "IFYP": "I Feel Your Pain",
    "ILU": "I Love You",
    "ILY": "I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMU": "I Miss You",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "IYKYK": "If You Know, You Know",
    "JK": "Just Kidding",
    "KISS": "Keep It Simple, Stupid",
    "L": "Loss",
    "L8R": "Later",
    "LDR": "Long Distance Relationship",
    "LMK": "Let Me Know",
    "LMAO": "Laughing My A** Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "M8": "Mate",
    "MFW": "My Face When",
    "MID": "Mediocre",
    "MRW": "My Reaction When",
    "MTE": "My Thoughts Exactly",
    "NVM": "Never Mind",
    "NRN": "No Reply Necessary",
    "NPC": "Non-Player Character",
    "OIC": "Oh I See",
    "OP": "Overpowered",
    "PITA": "Pain In The A**",
    "POV": "Point Of View",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A** Off",
    "RN": "Right Now",
    "SK8": "Skate",
    "STATS": "Your Sex And Age",
    "SUS": "Suspicious",
    "TBH": "To Be Honest",
    "TFW": "That Feeling When",
    "THX": "Thank You",
    "TIME": "Tears In My Eyes",
    "TLDR": "Too Long, Didn’t Read",
    "TNTL": "Trying Not To Laugh",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "W": "Win",
    "W8": "Wait...",
    "WB": "Welcome Back",
    "WTF": "What The F**k",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "WYD": "What You Doing?",
    "WYWH": "Wish You Were Here",
    "ZZZ": "Sleeping, Bored, Tired"
}


In [25]:
chat_word['U']

'You'

In [26]:
def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w.upper()in chat_word:
            new_text.append(chat_word[w.upper()])
        else:
            new_text.append(w)
    return " ".join(new_text)

In [27]:
chat_conversion("IMHO he is the best")

'In My Honest/Humble Opinion he is the best'

In [28]:
df['review'] = df['review'].apply(chat_conversion)

In [29]:
df['review']

0        one of the other reviewers has mentioned that ...
1        a wonderful little production br br the filmin...
2        i thought this was a wonderful way to spend Te...
3        basically theres a family where a little boy j...
4        petter matteis love in the Tears In My Eyes of...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    i am a catholic taught in parochial elementary...
49998    im going to have to disagree with the previous...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: object

Spelling Correction

In [30]:
from textblob import TextBlob

In [35]:
inc_text = 'ceertain connditioon dring several ggeneration aree moodifieed in the same manerr'

textblob = TextBlob(inc_text)

textblob.correct().string

'certain condition during several generation are modified in the same manner'

Removing stop words #a, the , of , are , my

In [36]:
from nltk.corpus import stopwords

In [37]:
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [41]:
def remove_stop_words(text):
    new_text = []
    for w in text.split():
        if w in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(w)
    x= new_text[:]
    new_text.clear()
    return " ".join(x)

In [42]:
remove_stop_words('I think my hand is broken and i am feeling nasuia . Would you take me home?')

'I think  hand  broken    feeling nasuia . Would  take  home?'

In [43]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend Te...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the Tears In My Eyes of...,positive


In [44]:
df['review'] = df['review'].apply(remove_stop_words)

KeyboardInterrupt: 

In [45]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend Te...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the Tears In My Eyes of...,positive


Handling emojies

either remove or conver to text

In [46]:
def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        u"\U00002700-\U000027BF"  # Dingbats
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U00002600-\U000026FF"  # Misc symbols
        u"\U00002B00-\U00002BFF"  # Misc symbols and arrows
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', text)

In [47]:
text = "Hello 😃🔥✨🚀 World!"
print(remove_emoji(text))  
# Output: Hello  World!

Hello  World!


In [48]:
import emoji
print(emoji.demojize('Python is 🔥☄️❤️‍🔥'))

Python is :fire::comet::heart_on_fire:


Tokenization

using split()

In [49]:
sent= ' I am going to delhi'
sent.split()

['I', 'am', 'going', 'to', 'delhi']

In [50]:
sent = ' I am going to delhi. I will stay there for 3 days. Let\'s hope the trip to be great'
sent.split('.')

[' I am going to delhi',
 ' I will stay there for 3 days',
 " Let's hope the trip to be great"]

In [51]:
# problem with split function
sent= ' I am going to delhi!'
sent.split()

['I', 'am', 'going', 'to', 'delhi!']

In [52]:
sent = ' I am going to delhi? I will stay there for 3 days. Let\'s hope the trip to be great'
sent.split('.')

[' I am going to delhi? I will stay there for 3 days',
 " Let's hope the trip to be great"]

Regular expression

In [53]:
sent3 = 'I am going to delhi!'
tokens= re.findall("[\w]+",sent3)
tokens

['I', 'am', 'going', 'to', 'delhi']

NLTK

In [54]:
from nltk.tokenize import word_tokenize, sent_tokenize

In [55]:
sent = 'I am going to delhi!'
word_tokenize(sent)

['I', 'am', 'going', 'to', 'delhi', '!']

In [56]:
sent = 'I have a Ph.D in A.I'
word_tokenize(sent)

['I', 'have', 'a', 'Ph.D', 'in', 'A.I']

Spacy

In [57]:
import spacy
nlp= spacy.load('en_core_web_sm')#best of these four method

In [58]:
doc = nlp(sent)

In [59]:
for token in doc:
    print(token)

I
have
a
Ph
.
D
in
A.I


Stemming

In [60]:
from nltk.stem.porter import PorterStemmer

In [61]:
ps = PorterStemmer()
def stem_words(text):
    return " ".join([ps.stem(word) for word in text.split()])

In [62]:
sample = " walk walking walks walked"
stem_words(sample)

'walk walk walk walk'

Lemmatization
#why use it? stemming is not valid word and not presentable to user but lemmatization is! 

In [63]:
import nltk
from nltk.stem import WordNetLemmatizer

# Download wordnet once
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

words = ["cats", "rocks", "running", "better"]

print([lemmatizer.lemmatize(w) for w in words])

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


['cat', 'rock', 'running', 'better']


In [64]:
import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp("cats rocks running better")

print([token.lemma_ for token in doc])


['cat', 'rock', 'run', 'well']
